# 02 · Modelado

Fase 4 de CRISP-DM. La lógica está en `src/bank_captacion/train.py` y `models.py`; este notebook presenta los resultados que generó el pipeline (`python -m bank_captacion.pipeline`). Todo se evalúa sobre el **train limpio (40.690 filas)** con la misma partición; el hold-out no se usa en esta fase.

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display, Markdown
from bank_captacion import config as cfg
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

def show(fig):
    display(Image(filename=str(cfg.FIGURES_DIR / fig)))

def table(name, **kw):
    df = pd.read_csv(cfg.TABLES_DIR / name, **kw)
    display(df)
    return df


## 1. Protocolo de validación

`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`: cada fold conserva la proporción de conversiones (~11,7 %). Todo el preprocesamiento (winsorización, imputación, escalado, codificación, SMOTE) está dentro del pipeline y se ajusta solo con los 4 folds de entrenamiento de cada iteración.

In [ ]:
table('cv_folds.csv')

## 2. Comparación de 10 algoritmos (modelo pre-contacto, sin `duration`)

Cada modelo representa una familia distinta vista en la materia. En esta primera comparación se usan pesos de clase en los modelos que los admiten.

In [ ]:
cv = pd.read_csv(cfg.TABLES_DIR / 'cv_results.csv')
display(cv[['model','label','family','roc_auc_fmt','pr_auc_fmt','lift@20_fmt','brier_fmt','gain@20_mean','fit_time_s_mean','predict_ms_per_1k_mean']])
show('model_cv_comparison.png')

**Lectura:**

- Los tres ensambles de árboles (LightGBM, XGBoost, Random Forest) forman el grupo superior (PR-AUC ≈ 0,45–0,46, ROC-AUC ≈ 0,80), casi 4 veces el piso del Dummy (0,117).
- La MLP queda cerca (0,43), pero sin ventaja y con menor interpretabilidad.
- La regresión logística (0,41) es el mejor modelo interpretable y el baseline de referencia: la brecha con boosting (~0,05 de PR-AUC) mide el valor de capturar no linealidades e interacciones.
- KNN y Naive Bayes rinden menos: KNN sufre con 67 dimensiones one-hot y NB con la suposición de independencia (hay variables redundantes como pdays/previous/poutcome).
- SVM RBF entrenado con 10.000 filas tiene buen ROC-AUC pero peor PR-AUC: ordena bien en general pero peor en la parte alta del ranking, que es la que importa.
- El árbol de profundidad 6 es el más débil pero es el más explicable (se usa como "reglas de negocio" en la Fase 5).

## 3. Codificación de categóricas en boosting: one-hot vs nativa

In [ ]:
table('encoding_comparison.csv')[['model','kind','pr_auc_fmt','roc_auc_fmt','lift@20_fmt']]

En LightGBM la codificación nativa es levemente mejor (y más simple: 24 columnas en vez de 67), así que se adopta. En XGBoost el one-hot es mejor por más que la tolerancia de 0,003, así que se mantiene one-hot.

## 4. Estrategias de desbalance (sobre los 3 mejores)

(a) sin tratamiento (el umbral se ajusta después), (b) pesos de clase, (c) SMOTE dentro del pipeline (solo sobre los folds de entrenamiento).

In [ ]:
imb = table('imbalance_results.csv')[['model','imbalance','kind','pr_auc_fmt','roc_auc_fmt','lift@20_fmt','brier_fmt']]
show('imbalance_comparison.png')

**Resultado:** las tres estrategias quedan dentro de un desvío estándar entre sí: el desbalance (11,7 %) es moderado y los modelos de ranking no lo necesitan. Pero **pesos y SMOTE empeoran la calibración** (Brier de LightGBM: 0,081 sin tratamiento vs 0,144 con pesos), porque inflan artificialmente la probabilidad de la clase positiva. Como el umbral y el beneficio esperado se calculan con probabilidades, se elige **sin tratamiento + ajuste de umbral**. Regla aplicada: si la diferencia de PR-AUC es menor a 0,003 se prefiere la estrategia que no distorsiona las probabilidades. SMOTE además agrega costo de cómputo y genera clientes sintéticos sobre variables one-hot, que no tienen interpretación (interpolar entre "married" y "single").

## 5. Tuning con Optuna (2 mejores modelos)

Sampler TPE con semilla 42, 60 trials por modelo, tope de 15 minutos, objetivo = PR-AUC media en los mismos 5 folds.

In [ ]:
tun = table('tuning_summary.csv')
for _, r in tun.iterrows():
    print(r['model'], json.loads(r['best_params']))
show('tuning_convergence_lightgbm.png')
show('tuning_convergence_xgboost.png')

La mejora por tuning es chica (≈ +0,007 en LightGBM y +0,011 en XGBoost): los valores por defecto ya eran razonables y el techo lo pone la información disponible antes de llamar, no el algoritmo. Advertencia: el PR-AUC del mejor trial es optimista porque se eligió mirando esos mismos folds; la estimación sin sesgo es la del hold-out (Fase 5).

## 6. Selección final: matriz de decisión

Criterios y pesos: PR-AUC 35 %, lift@20 % 25 %, estabilidad (desvío de PR-AUC entre folds) 15 %, interpretabilidad 15 %, velocidad de inferencia 10 %. Cada criterio se normaliza 0–1 entre candidatos.

In [ ]:
dm = table('decision_matrix.csv')[['candidate','pr_auc','pr_auc_std','roc_auc','lift@20','brier','predict_ms_per_1k','interpretability','weighted_score']]

Gana **LightGBM tuneado** (sin tratamiento de desbalance, categorías nativas): mejor PR-AUC y lift, inferencia rápida. La regresión logística es la más estable e interpretable, pero pierde ~0,05 de PR-AUC: se conserva como baseline y como contraste de explicabilidad. La falta de interpretabilidad de LightGBM se compensa con SHAP (Fase 5).

## 7. Umbral operativo

In [ ]:
thr = json.loads((cfg.TABLES_DIR / 'threshold_choice.json').read_text())
print(json.dumps(thr, indent=2))
show('threshold_selection.png')
table('sensitivity_oof.csv')

- El umbral que **maximiza el beneficio** con V = 20 y C = 1 es **0,050**, igual al umbral teórico C/V: consistente con probabilidades bien calibradas. Contacta al ~62 % de los clientes y captura el 89 % de las conversiones.
- El umbral que **maximiza F1** es 0,215: contacta solo al ~13 % y deja muchas conversiones rentables sin llamar. F1 pondera igual precisión y recall, sin considerar que una conversión vale 20 veces lo que cuesta una llamada; por eso su beneficio es menor.
- La sensibilidad muestra que el umbral depende del ratio V/C: con V/C = 5 conviene contactar al 16 %; con V/C = 50, a casi todos.

## 8. Efecto de `duration` (leakage) en CV

In [ ]:
table('leakage_cv.csv')[['feature_set','roc_auc_fmt','pr_auc_fmt','lift@20_fmt']]

Agregar `duration` sube el ROC-AUC de 0,81 a 0,94 y el PR-AUC de 0,47 a 0,64. Es una mejora **ilusoria**: esa información no existe cuando hay que decidir a quién llamar.